In [1]:
# 01 · EASY 집중 실험 CONFIG
CFG = {
    # GitHub 저장 · 기존 결과 보존, Easy 전용 Release
    'run_name': 'moveboxes_easy_lab_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'main',
    'output_root': '/content/moveboxes_runs',
    'source_run_name': 'moveboxes_stage_act_v1',
    'warm_start': True,

    # T4 / 설치 / 데이터
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 반복 예산 · 2,000회마다 실제 평가, 최대 20,000회
    'block_iters': 2000,
    'max_blocks': 10,
    'development_episodes': 8,
    'target_accuracy': 0.95,
    'success_streak': 2,
    'plateau_blocks': 4,

    # 작은 모델 · 실행 조건으로 학습, 첫 집기 구간 30% 별도 표집
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'amp': True,
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,
    'first_pick_fraction': 0.3,

    # 별도 시드와 동일한 200스텝 평가 제한
    'tuning_seed_start': 50000,
    'eval_seed_start': 70000,
    'final_episodes': 100,
    'test_seed_start': 40000,
    'test_episodes': 8,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},

    # 실행 / 출력
    'ensemble_candidates': [1, 4],
    'ensemble_window': 4,
    'temporal_decay': 0.25,
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'console_interval_seconds': 30,
    'team': 'my-team',

}


In [2]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
for name in ('build_easy_notebook','easy_lab'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from build_easy_notebook import CONFIG as EASY_DEFAULTS
CFG = dict(EASY_DEFAULTS, **CFG)
from easy_lab import EasyLab, source_bundle
experiment = EasyLab(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


사용 코드: 7d49697cfee77ffab1303bed0dc2f53350a593be
코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.


In [3]:
# 03 · GitHub 인증/복원
experiment.connect()
experiment.report()


GitHub 토큰 입력 (이 런타임에서만 사용): ··········
GitHub 복원: 0 files
로컬 작업 경로: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark
실행 노트북 사본: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/moveboxes_easy_colab.ipynb
새 런타임에서는 01~03 셀로 결과를 복원하고, 학습·평가는 04~05 셀 준비 후 실행합니다.
GitHub 백업 완료: common (46 files)
단계 ACT: 학습한 완료/복구 판단 + 집기/운반/놓기 행동, 별도 실험
상태: not_started


{'blocks': [], 'status': 'not_started'}

In [4]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


실행: nvidia-smi
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/sessions/20260914_011746_3a3393/gpu.log
완료
실행: git clone https://github.com/marso-robotics/berlin-marso-hackathon.git /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/sessions/20260914_011746_3a3393/commands.log
완료
실행: /usr/bin/python3 -m pip install mani-skill==3.0.1 sapien==3.0.3 diffusers==0.38.0 hydra-core omegaconf gymnasium tyro h5py kagglehub tensorboard matplotlib transforms3d imageio[ffmpeg]
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/sessions/20260914_011746_3a3393/install.log
완료
실행: /usr/bin/python3 -m pip install -e /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/sessions/20260914_011746_3a3393/install_repo.log
완료
설치 완료. 다음 데이터·GPU 확인 셀을 실행하세요.


In [5]:
# 05 · Easy GPU 확인 / 기존 성공 시연과 모델 가져오기
experiment.check_runtime()
experiment.prepare()


실행: /usr/bin/python3 -c import torch
from warehouse_sort.utils import compose_cfg, make_env
assert torch.cuda.is_available(), 'T4 GPU runtime required'
cfg = compose_cfg(['difficulty=easy', 'num_envs=1'])
env, _ = make_env(cfg, 'state', cfg.randomization, num_envs=1, render_mode='rgb_array')
try:
    obs, _ = env.reset(seed=42)
    assert tuple(obs.shape) == (1,54)
    env.step(torch.zeros((1,4),device='cuda'))
    assert env.render() is not None
    print('Easy GPU/state/render OK')
finally:
    env.close()

전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/sessions/20260914_011746_3a3393/easy_gpu_check.log
완료
GitHub 데이터 다운로드: state (65.4 MiB)
easy 200 demos
medium 200 demos
hard 200 demos
GitHub 백업 완료: common (53 files)
GitHub 백업 완료: easy (20 files)
Easy 성공 시연 16개 준비 완료. 반복 학습 셀을 실행하세요.


In [6]:
# 06 · EASY 반복 학습 · 중단 후 같은 셀을 다시 실행하면 이어서 진행
history = experiment.run_blocks()


실행: /usr/bin/python3 stage_train.py /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/stage_train_job.json
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/train_block_01.log
753/20000 loss=0.0470 25.1 it/s ETA=12.8 min
1547/20000 loss=0.0141 25.8 it/s ETA=11.9 min
GitHub 백업 완료: easy (30 files)
완료
실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/eval_job.json
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/dev_b01_w1.log
GitHub 백업 완료: easy (34 files)
GitHub 백업 완료: easy (34 files)
GitHub 백업 완료: easy (34 files)
완료
실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/eval_job.json
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/dev_b01_w4.log
GitHub 백업 완료: easy (37 files)
GitHub 백업 완료: easy (37 files)
GitHub 백업 완료: easy (37 files)
완료
GitHub 백업 완료: easy (40 files)
Easy 2000회: 첫 집기 100% / 두 상자 정답 38% / 분류 68.8%
실행: /u

In [7]:
# 07 · 최고 모델 8회 별도 테스트 + 영상 + 2회 행동 기록
experiment.test("easy")
experiment.diagnose("easy")


실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/eval_job.json
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/test_metrics.log
GitHub 백업 완료: easy (66 files)
GitHub 백업 완료: easy (66 files)
GitHub 백업 완료: easy (66 files)
완료
[easy] 빠른 테스트 8회: 100.0%
결과: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/test_metrics.json / 모델: block_03.pt
빠른 테스트는 최종 가중 점수에 합산하지 않습니다.
실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/test_video_job.json
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/test_video.log
완료


[easy] 8회 중 1개 이상 정답 분류 100.0% / 2개 이상 100.0% / 1개만 0.0%
첫 안정적 집기 이후 두 번째 집기 사이클 관측: 100.0%
집기 사이클은 같은 상자를 다시 잡는 경우도 포함합니다. 실제 분류 성과는 위 정답 개수로 확인하세요.
에피소드별 집기 시점·집게 명령 전환: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/test_metrics.json
GitHub 백업 완료: easy (71 files)
실행: /usr/bin/python3 colab_trace.py /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/trace_job.json
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/trace.log
GitHub 백업 완료: easy (75 files)
완료
GitHub 백업 완료: easy (75 files)
[easy] 2회 상태·행동 진단 저장: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/traces/0ed2513484214be5


In [8]:
# 08 · 학습 곡선과 최고 결과
experiment.report()


 2000회 | 집기 100% | 두 상자 정답 38% | 분류 68.8%
 4000회 | 집기 0% | 두 상자 정답 0% | 분류 0.0%
 6000회 | 집기 100% | 두 상자 정답 100% | 분류 100.0%
 8000회 | 집기 100% | 두 상자 정답 100% | 분류 100.0%
상태: target_reached


{'blocks': [{'block': 1,
   'iteration': 2000,
   'best': {'checkpoint': '/content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/checkpoints/block_01.pt',
    'chunk': 1,
    'steps': 1,
    'score': 0.6875,
    'policy_config': {'model_config': {'state_dim': 54,
      'history': 16,
      'chunk_size': 16,
      'width': 128,
      'heads': 4,
      'layers': 2,
      'latent_dim': 16},
     'temporal_decay': 0.25,
     'ensemble_window': 1,
     'gate_threshold': 0.65,
     'stage_threshold': 0.6,
     'act_horizon': 1,
     'num_inference_steps': 1},
    'checkpoint_sha256': '783b8e153b89a2e0bbf1ff8a35b6d8733d43887ce21af85d57325d9f1fec49ee',
    'first_grasp_rate': 1.0,
    'both_sorted_rate': 0.375,
    'block': 1,
    'iteration': 2000},
   'improved': True,
   'trials': [{'checkpoint': '/content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/checkpoints/block_01.pt',
     'chunk': 1,
     'steps': 1,
     'score': 0.6875,
     'policy_config': {'model_config': {'state_

In [9]:
# 09 · 목표 달성 모델의 별도 시드 최종 평가
experiment.final_evaluation()


실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/eval_job.json
전체 로그: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/easy/metrics.log
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78 files)
GitHub 백업 완료: easy (78

{'sort_accuracy': 1.0,
 'mean_sorted': 2.0,
 'all_placed_rate': 1.0,
 'mean_steps': 120.0,
 'mis_sort_rate': 0.0,
 'n_episodes': 100,
 'episodes': [{'seed': 70000,
   'n_episodes': 1,
   'num_parcels': 2,
   'sort_accuracy': 1.0,
   'mean_sorted': 2.0,
   'all_placed_rate': 1.0,
   'mean_steps': 120.0,
   'mis_sort_rate': 0.0,
   'next_pick': {'stable_grasp_onsets': [30, 89],
    'stable_grasp_cycles': 2,
    'gripper_sign_reversals': 4,
    'observed_steps': 199,
    'stages': {'visits': {'pick': 74, 'carry': 21, 'place': 26, 'done': 78},
     'transitions': 6,
     'recovery_decisions': 0,
     'rejected_proposals': 0}}},
  {'seed': 70001,
   'n_episodes': 1,
   'num_parcels': 2,
   'sort_accuracy': 1.0,
   'mean_sorted': 2.0,
   'all_placed_rate': 1.0,
   'mean_steps': 120.0,
   'mis_sort_rate': 0.0,
   'next_pick': {'stable_grasp_onsets': [30, 89],
    'stable_grasp_cycles': 2,
    'gripper_sign_reversals': 4,
    'observed_steps': 199,
    'stages': {'visits': {'pick': 74, 'carry'

In [10]:
# 10 · 평가 완료한 Easy 모델만 패키징
experiment.package()


저장한 패키지: /content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/checkpoint_package_20260914_020418.zip
GitHub 백업 완료: common (54 files)


'/content/moveboxes_runs/moveboxes_easy_lab_v1_benchmark/checkpoint_package_20260914_020418.zip'